In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import pprint

# A set of fixed input
a_value = 1.0
n_psa = 10000
seed = 25

sample_size_now = 397
sample_size_new = 619

wtp = 150000

max_cycle = 200
# Define the cycle range
cycle_range = np.arange(0, max_cycle + 1)  # 0 to 200 inclusive

def get_discount_factor(cycle_range, dr=0.03, cycles_per_year=16):
    """
    Calculates the discount factor for each cycle based on the discount rate.

    Args:
        cycle_range (array-like): The range of cycles (e.g., np.arange(0, 201)).
        dr (float): The yearly discount rate (default is 0.03, or 3%).
        cycles_per_year (int): The number of cycles in a year (default is 16).

    Returns:
        np.ndarray: An array of discount factors corresponding to the cycle range.
    """
    # Convert yearly discount rate to per-cycle discount rate
    discount_rate_cycle = (1 + dr) ** (1 / cycles_per_year) - 1

    # Calculate the discount factor for each cycle
    discount_factor = 1 / (1 + discount_rate_cycle) ** cycle_range

    return discount_factor

discount_factor = get_discount_factor(cycle_range)

# Hazard ratio for Combo versus Chemo
hr_params = {
    "pfs": {"hr": 0.49, "low": 0.38, "high": 0.63},
    "os": {"hr": 0.60, "low": 0.45, "high": 0.79}
}

# Define the utility dictionary
utility = {
    "stable": 0.75,  # Utility for stable state
    "prog": 0.59     # Utility for progression state
}

# Define the cost dictionary
cost = {
    # Monitoring costs
    "monitoring_stable": 464.85 * 3,
    "monitoring_prog": 1075.49 * 3,

    # Parameters for drug costs
    "surface": 1.86,
    "admin_first": 158.7,
    "admin_sub": 33.6,
    "terminal": 16441.83,
    "price_premetrxed": 7.49,
    "price_cisplatin": 0.18,
    "price_carboplatin": 0.05,
    "dose_premetrxed": 500,
    "dose_cisplatin": 75,
    "dose_carboplatin": 550,
    "dose_drug": 200,

    # Calculated costs
    "intro_cost": (
        1.86 * 500 * 7.49 +
        0.277 * 1.86 * 75 * 0.18 +
        (1 - 0.277) * 550 * 0.05
    ),
    "maintain_cost_chemo": 1.86 * 500 * 7.49
}

In [ ]:
value_based_price_loc = "/content/drive/MyDrive/Colab Notebooks/01_Research/01_Local_Confirmatory_RCT/03_output/02_value_price"
value_based_price = pd.read_csv(f"{value_based_price_loc}/value_price.csv").iloc[0,0]

surv_params_path = "/content/drive/MyDrive/Colab Notebooks/01_Research/01_Local_Confirmatory_RCT/03_output/01_surv_params"
surv_params = pd.read_csv(f"{surv_params_path}/surv_params.csv")
case_params = {
    row["case_name"]: {
        "intercept": row["intercept"],
        "log_scale": row["log_scale"]
    }
    for _, row in surv_params.iterrows()
}

In [ ]:
def hr_log_transform(hr: float,
                     hr_lower: float,
                     hr_upper: float,
                     a_value: float=a_value) -> dict:
    """
    Compute the mean and standard deviation of the log-transformed hazard ratio (HR).

    Parameters:
    hr (float): Hazard ratio estimate.
    ci_lower (float): Lower bound of the confidence interval.
    ci_upper (float): Upper bound of the confidence interval.

    Returns:
    tuple: (mean of log(HR), standard deviation of log(HR))
    """
    log_hr = np.log(hr)
    log_ci_lower = np.log(hr_lower)
    log_ci_upper = np.log(hr_upper)

    # Approximate standard deviation using CI width
    sigma_log_hr_old = (log_ci_upper - log_ci_lower) / (2 * norm.ppf(1 - 0.05 / 2))
    sigma_log_hr_updated = np.sqrt(sigma_log_hr_old**2 / a_value)

    return {"mu_log_hr": log_hr,
            "sigma_log_hr": sigma_log_hr_old,
            "sigma_log_hr_updated": sigma_log_hr_updated}

In [ ]:
def get_simparams(
    a_value: float = a_value,
    sample_size_now: int = sample_size_now,
    sample_size_new: int = sample_size_new,
    hr_params: dict = hr_params,
    n_psa: int = n_psa,
    seed_for_prior: int = seed
) -> dict:
    """
    Generate prior and posterior hazard ratio (HR) samples for PFS and OS.

    Returns:
        dict: A nested dictionary with keys ["prior"]["pfs"], ["prior"]["os"],
              ["posterior"]["pfs"], ["posterior"]["os"] each containing arrays of HR samples.
    """
    from numpy.random import default_rng

    # RNG seeded for reproducibility of the "prior" samples
    rng_prior = default_rng(seed_for_prior)

    # Dictionary to hold final results
    res = {
        "prior": {},
        "posterior": {}
    }

    # Compute log-HR parameters (mu and sigma) for PFS and OS
    log_hr_dict = {
        "pfs": hr_log_transform(
            hr_params["pfs"]["hr"],
            hr_params["pfs"]["low"],
            hr_params["pfs"]["high"],
            a_value=a_value
        ),
        "os": hr_log_transform(
            hr_params["os"]["hr"],
            hr_params["os"]["low"],
            hr_params["os"]["high"],
            a_value=a_value
        )
    }

    # Iterate over PFS and OS
    for outcome, params in log_hr_dict.items():
        prior_mean       = params["mu_log_hr"]
        prior_sd         = params["sigma_log_hr"]
        prior_sd_updated = params["sigma_log_hr_updated"]

        # Precision of the *original* prior distribution
        prior_precision = 1.0 / (prior_sd**2)

        # Draw from the prior distribution
        # prior_mean_samples         = rng_prior.normal(prior_mean, prior_sd,         size=n_psa)
        prior_mean_updated_samples = rng_prior.normal(prior_mean, prior_sd_updated, size=n_psa)

        # Exponentiate to get HR samples (prior)
        # prior_hr_arr = np.exp(prior_mean_samples)
        prior_hr_arr = np.exp(prior_mean_updated_samples)

        # Population variance of the log HR scaled by sample_size_now
        pop_var = (prior_sd**2) * sample_size_now
        # The sample variance for the sample mean in new data
        sample_var = pop_var / sample_size_new
        sample_precision = 1.0 / sample_var

        # For each PSA replicate, simulate new data
        # Generate a (n_psa*sample_size_new) array
        new_data_samples = rng_prior.normal(
            loc = prior_mean_updated_samples.reshape(n_psa, 1),
            scale = np.sqrt(sample_var),
            size = (n_psa, sample_size_new)
        )

        # Compute the sample mean for each replicate
        X_bar = new_data_samples.mean(axis=1)

        # Normal-Normal Bayesian update for each replicate
        post_mean = (prior_precision * prior_mean + sample_precision * X_bar) / (prior_precision + sample_precision)
        post_sd = prior_sd * np.sqrt(sample_var / (sample_var + prior_sd**2))

        # Draw posterior samples for each replicate
        # Generate an (n_psa*n_psa) array: each row i contains n_psa samples from N(post_mean[i], post_sd)
        post_samples = rng_prior.normal(
            loc=post_mean.reshape(n_psa, 1),
            scale=post_sd,
            size=(n_psa, n_psa)
        )

        post_hr_matrix = np.exp(post_samples)  # Convert to HR values
        post_hr_dict = {i: post_hr_matrix[i, :] for i in range(n_psa)}  # Store as dictionary entry

        # post_hr_dict = {}  # Initialize dictionary for posterior samples

        # # Sample each "PSA replicate" of the posterior
        # for i in range(n_psa):

        #     # Draw one "true" mean from updated prior
        #     prior_sample_mean_i = prior_mean_updated_samples[i]

        #     # Suppose the population variance of the log(HR) is scaled by sample_size_now
        #     # pop_var = (prior_sd_updated**2) * sample_size_now
        #     pop_var = (prior_sd**2) * sample_size_now
        #     # pop_sd  = np.sqrt(pop_var)

        #     # The sample variance of the (sample mean) around that "true" mean
        #     sample_var = pop_var / sample_size_new

        #     # Simulate a sample mean X̄ for the new data
        #     X_bar_i = rng_prior.normal(prior_sample_mean_i, np.sqrt(sample_var), size=sample_size_new).mean()

        #     # Combine prior and sample by normal-normal update
        #     sample_precision = 1.0 / sample_var

        #     ## mean
        #     post_mean_1 = prior_precision * prior_mean + sample_precision * X_bar_i
        #     post_mean_2 = prior_precision + sample_precision
        #     post_mean_i = post_mean_1 / post_mean_2

        #     ## variance
        #     post_sd_i = prior_sd * np.sqrt(sample_var / (sample_var + prior_sd**2))

        #     ## posterior log hr
        #     post_mean_arr = rng_prior.normal(post_mean_i, post_sd_i, size=n_psa)

        #     # Store exponentiated posterior mean
        #     post_mean_arr = rng_prior.normal(post_mean_i, post_sd_i, size=n_psa)
        #     post_hr_dict[i] = np.exp(post_mean_arr)  # Store as dictionary entry

        # Place results into the final dictionary
        res["prior"][outcome] = prior_hr_arr
        res["posterior"][outcome] = post_hr_dict

    return res

In [ ]:
def get_nmb(
    hr_pair: dict,
    case_params: dict,
    price_drug: float,
    wtp: float,
    utility: dict,
    cost: dict,
    cycle_range: np.ndarray,
    discount_factor: np.ndarray
) -> tuple:
    """
    Calculate net monetary benefit (NMB) and related economic outcomes
    for combination therapy vs. standard chemotherapy, given:
      - Hazard ratios for PFS and OS
      - Model parameters for cost, utility, and survival intercept/log_scale
      - Price of the drug and willingness-to-pay (WTP).

    Parameters
    ----------
    hr_pair : dict
        A dictionary with hazard ratios for PFS and OS, e.g.:
           {"pfs": float, "os": float}

    case_params : dict
        A dictionary containing intercept and log_scale (if needed) for
        survival in chemo vs. combo arms (though here we only use chemo's
        intercept/log_scale and apply the HR).
        Example:
        {
            "pfs_chemo": {"intercept": ..., "log_scale": ...},
            "pfs_combo": {"intercept": ..., "log_scale": ...},
            "os_chemo":  {"intercept": ..., "log_scale": ...},
            "os_combo":  {"intercept": ..., "log_scale": ...}
        }

    price_drug : float
        The price of the new drug per dose.

    wtp : float
        Willingness-to-pay threshold (per QALY).

    utility : dict
        A dictionary containing utility values for "stable" and "progression" states.
        Example: {"stable": 0.75, "prog": 0.59}

    cost : dict
        A dictionary with cost parameters, e.g.:
        {
            "monitoring_stable": ...,
            "monitoring_prog": ...,
            "admin_first": ...,
            "admin_sub": ...,
            "terminal": ...,
            "intro_cost": ...,
            "maintain_cost_chemo": ...,
            "dose_drug": ...
        }

    cycle_range : np.ndarray
        The range of cycles for the simulation (e.g., np.arange(0, max_cycle + 1)).

    discount_factor : np.ndarray
        An array of discount factors corresponding to each cycle in cycle_range.

    Returns
    -------
    tuple
        (
            Avg_cost_per_month_chemo,
            Avg_cost_per_month_combo,
            MB_chemo,
            MB_combo,
            NMB,
            ICER
        )

        - Avg_cost_per_month_chemo / Avg_cost_per_month_combo :
          The average cost per month of chemo/combo (total cost / #cycles,
          scaled to 1 month if each cycle is < 1 month).
        - MB_chemo / MB_combo : Monetary benefit (WTP * QALYs - Costs).
        - NMB : Net Monetary Benefit = MB_combo - MB_chemo.
        - ICER : Incremental Cost-Effectiveness Ratio
                 = (Cost_combo - Cost_chemo) / (QALYs_combo - QALYs_chemo).
    """

    # 1. Survival simulation function
    def simulate_survival(cycle_range: np.ndarray, intercept: float, log_scale: float, hr=None):
        """
        Computes survival probabilities for each cycle.
        If hr is given, it exponentiates the 'control' survival curve by hr.
        """
        # Convert cycle index to approximate months or weeks; here it's 3/4 of a month per cycle
        adjusted_cycle = cycle_range * (3 / 4)

        # Baseline (chemo) survival curve, using a log-normal approximation:
        #   surv_ctrl = 1 - Φ( (log(time) - μ) / σ )
        # We add a tiny number to avoid log(0).
        surv_ctrl = 1 - norm.cdf(
            (np.log(adjusted_cycle + 1e-10) - intercept) / np.exp(log_scale)
        )
        # Force the very last cycle's survival prob to 0
        surv_ctrl[-1] = 0

        # If hazard ratio is provided, apply it by exponentiating the control survival
        # (This is a simplified approach; be sure it matches your modeling strategy.)
        if hr is not None:
            surv_prob = surv_ctrl ** hr
        else:
            surv_prob = surv_ctrl

        # Force the last cycle's survival prob to 0
        surv_prob[-1] = 0
        return surv_prob

    # 2. Simulate PFS & OS for chemotherapy (chemo)
    pfs_chemo = simulate_survival(
        cycle_range,
        intercept=case_params["pfs_chemo"]["intercept"],
        log_scale=case_params["pfs_chemo"]["log_scale"]
    )
    os_chemo = simulate_survival(
        cycle_range,
        intercept=case_params["os_chemo"]["intercept"],
        log_scale=case_params["os_chemo"]["log_scale"]
    )

    # 3. Simulate PFS & OS for combination, applying hazard ratio from hr_pair
    pfs_combo = simulate_survival(
        cycle_range,
        intercept=case_params["pfs_chemo"]["intercept"],
        log_scale=case_params["pfs_chemo"]["log_scale"],
        hr=hr_pair['pfs']
    )
    os_combo = simulate_survival(
        cycle_range,
        intercept=case_params["os_chemo"]["intercept"],
        log_scale=case_params["os_chemo"]["log_scale"],
        hr=hr_pair['os']
    )

    # 4. Define states for chemo
    prog_chemo = np.maximum(os_chemo - pfs_chemo, 0)
    stable_chemo = pfs_chemo
    dead_chemo = 1 - os_chemo

    # Force last cycle’s stable/prog to 0, dead to 1
    stable_chemo[-1] = 0
    prog_chemo[-1] = 0
    dead_chemo[-1] = 1

    # 5. QALYs for chemo
    # Each cycle ~ 3/4 of a month, so multiply utility * (3/(4*12)) to get QALYs per cycle
    qalys_chemo = np.sum(
        stable_chemo * (utility["stable"] * 3/(4*12)) * discount_factor +
        prog_chemo   * (utility["prog"]   * 3/(4*12)) * discount_factor
    )

    # 6. Costs for chemo
    # 6a. Monitoring cost
    monitoring_chemo = (
        cost["monitoring_stable"] * stable_chemo
      + cost["monitoring_prog"]   * prog_chemo
    )

    # 6b. Administration cost for first 37 cycles
    admin_chemo = np.zeros(len(cycle_range))
    admin_chemo[:37] = stable_chemo[:37] * (cost["admin_first"] + cost["admin_sub"])

    # 6c. Terminal cost upon death
    final_chemo = np.concatenate(([dead_chemo[0]], np.diff(dead_chemo))) * cost["terminal"]

    # 6d. Drug treatment cost (induction vs maintenance)
    treatment_chemo = np.zeros(len(cycle_range))
    treatment_chemo[:4]    = cost["intro_cost"] * stable_chemo[:4]            # Induction (~4 cycles)
    treatment_chemo[4:36]  = cost["maintain_cost_chemo"] * stable_chemo[4:36] # Maintenance (~32 cycles)

    # Total discounted costs
    costs_chemo = np.sum(
        (monitoring_chemo + admin_chemo + final_chemo + treatment_chemo)
        * discount_factor
    )

    # 7. Define states for combo
    prog_combo = np.maximum(os_combo - pfs_combo, 0)
    stable_combo = pfs_combo
    dead_combo = 1 - os_combo

    stable_combo[-1] = 0
    prog_combo[-1] = 0
    dead_combo[-1] = 1

    # 8. QALYs for combo
    qalys_combo = np.sum(
        stable_combo * (utility["stable"] * 3/(4*12)) * discount_factor +
        prog_combo   * (utility["prog"]   * 3/(4*12)) * discount_factor
    )

    # 9. Costs for combo
    # 9a. Induction & maintenance drug costs
    drug_cost_intro    = cost["dose_drug"] * price_drug + cost["intro_cost"]
    drug_cost_maintain = cost["dose_drug"] * price_drug + cost["maintain_cost_chemo"]

    # 9b. Monitoring
    monitoring_combo = (
        cost["monitoring_stable"] * stable_combo
      + cost["monitoring_prog"]   * prog_combo
    )

    # 9c. Administration cost for first 37 cycles
    admin_combo = np.zeros(len(cycle_range))
    admin_combo[:37] = stable_combo[:37] * (cost["admin_first"] + cost["admin_sub"])

    # 9d. Terminal cost
    final_combo = np.concatenate(([dead_combo[0]], np.diff(dead_combo))) * cost["terminal"]

    # 9e. Induction (first 4 cycles) & maintenance (cycles 4-35)
    treatment_combo = np.zeros(len(cycle_range))
    treatment_combo[:4]   = drug_cost_intro * stable_combo[:4]
    treatment_combo[4:36] = drug_cost_maintain * stable_combo[4:36]

    # Total discounted costs
    costs_combo = np.sum(
        (monitoring_combo + admin_combo + final_combo + treatment_combo)
        * discount_factor
    )

    # 10. Calculate cost-effectiveness outcomes
    # Average cost per month (since each cycle is 3 weeks, we do (4/3) to scale up to monthly from cycle-avg).
    Avg_cost_per_month_chemo = (costs_chemo / len(cycle_range)) * (4/3)
    Avg_cost_per_month_combo = (costs_combo / len(cycle_range)) * (4/3)

    # Monetary benefit for each arm
    MB_chemo = wtp * qalys_chemo - costs_chemo
    MB_combo = wtp * qalys_combo - costs_combo

    # Net monetary benefit (combo - chemo)
    NMB = MB_combo - MB_chemo

    # Incremental cost-effectiveness ratio
    delta_cost = costs_combo - costs_chemo
    delta_qaly = qalys_combo - qalys_chemo
    if delta_qaly == 0:
        ICER = np.inf if delta_cost > 0 else 0
    else:
        ICER = delta_cost / delta_qaly

    return (
        Avg_cost_per_month_chemo,
        Avg_cost_per_month_combo,
        MB_chemo,
        MB_combo,
        NMB,
        ICER
    )

In [ ]:
def compute_psa_nmb(
    a_value: float = a_value,
    sample_size_now: int = sample_size_now,
    sample_size_new: int = sample_size_new,
    n_psa: int = n_psa,
    seed_for_prior: int = seed,
    price_drug: float = value_based_price,
    wtp: float = wtp,
    case_params: dict = case_params,
    hr_params: dict = hr_params,
    utility: dict = utility,
    cost: dict = cost,
    cycle_range: np.ndarray = cycle_range,
    discount_factor: np.ndarray = discount_factor
) -> dict:
    """
    Simulates cost and net monetary benefit (NMB) metrics for prior and posterior
    hazard-ratio draws, returning each result as a 1D NumPy array of length n_psa.

    For the posterior, the get_simparams function returns a dictionary of
    n_psa arrays (each subarray is length n_psa). We compute the average
    across each subarray, so each posterior result ends up as an n_psa array.
    """

    # 1) Get hazard-ratio draws from get_simparams
    psa_params = get_simparams(
        a_value=a_value,
        sample_size_now=sample_size_now,
        sample_size_new=sample_size_new,
        hr_params=hr_params,
        n_psa=n_psa,
        seed_for_prior=seed_for_prior
    )
    # Structure might look like:
    # {
    #   "prior": {
    #       "pfs": np.ndarray(n_psa),
    #       "os":  np.ndarray(n_psa)
    #   },
    #   "posterior": {
    #       "pfs": {0: np.ndarray(n_psa), 1: np.ndarray(n_psa), ...},
    #       "os":  {0: np.ndarray(n_psa), 1: np.ndarray(n_psa), ...}
    #   }
    # }

    # 2) Prepare arrays for prior results
    Avg_cost_per_month_chemo_prior = np.zeros(n_psa)
    Avg_cost_per_month_combo_prior = np.zeros(n_psa)
    MB_chemo_prior = np.zeros(n_psa)
    MB_combo_prior = np.zeros(n_psa)

    # 3) Prepare arrays for posterior results (length n_psa).
    #    Each index i will store the *average* across the sub-distribution.
    Avg_cost_per_month_chemo_post = np.zeros(n_psa)
    Avg_cost_per_month_combo_post = np.zeros(n_psa)
    MB_chemo_post = np.zeros(n_psa)
    MB_combo_post = np.zeros(n_psa)

    # -------------------------------------------------------------------------
    # 4) Compute Prior-based Results: straightforward, 1 hazard ratio per iteration
    # -------------------------------------------------------------------------
    for i in range(n_psa):
        hr_pair_prior = {
            "pfs": psa_params["prior"]["pfs"][i],
            "os":  psa_params["prior"]["os"][i]
        }
        (
            Avg_cost_per_month_chemo_prior[i],
            Avg_cost_per_month_combo_prior[i],
            MB_chemo_prior[i],
            MB_combo_prior[i],
            _nmb,
            _icer
        ) = get_nmb(
            hr_pair=hr_pair_prior,
            case_params=case_params,
            price_drug=price_drug,
            wtp=wtp,
            utility=utility,
            cost=cost,
            cycle_range=cycle_range,
            discount_factor=discount_factor
        )

    # -------------------------------------------------------------------------
    # 5) Compute Posterior-based Results: for each i, get arrays of length n_psa,
    #    then compute average across those sub-arrays.
    # -------------------------------------------------------------------------
    for i in range(n_psa):
        # Each of these is shape (n_psa,)
        post_pfs_arr = psa_params["posterior"]["pfs"][i]
        post_os_arr  = psa_params["posterior"]["os"][i]

        # Prepare subarrays to store each "mini-PSA" iteration's NMB results
        # for the i-th posterior
        chemo_cost_sub   = np.zeros(n_psa)
        combo_cost_sub   = np.zeros(n_psa)
        MB_chemo_sub     = np.zeros(n_psa)
        MB_combo_sub     = np.zeros(n_psa)

        # For j in range(n_psa), call get_nmb
        for j in range(n_psa):
            hr_pair_post = {
                "pfs": post_pfs_arr[j],
                "os":  post_os_arr[j]
            }
            (
                chemo_cost_sub[j],
                combo_cost_sub[j],
                MB_chemo_sub[j],
                MB_combo_sub[j],
                _nmb,
                _icer
            ) = get_nmb(
                hr_pair=hr_pair_post,
                case_params=case_params,
                price_drug=price_drug,
                wtp=wtp,
                utility=utility,
                cost=cost,
                cycle_range=cycle_range,
                discount_factor=discount_factor
            )

        # Now take the *average* across those n_psa draws
        Avg_cost_per_month_chemo_post[i] = np.mean(chemo_cost_sub)
        Avg_cost_per_month_combo_post[i] = np.mean(combo_cost_sub)
        MB_chemo_post[i] = np.mean(MB_chemo_sub)
        MB_combo_post[i] = np.mean(MB_combo_sub)

    # -------------------------------------------------------------------------
    # 6) Return the final structure (all arrays of length n_psa)
    # -------------------------------------------------------------------------
    return {
        "Avg_cost_per_month_chemo_prior": Avg_cost_per_month_chemo_prior,
        "Avg_cost_per_month_combo_prior": Avg_cost_per_month_combo_prior,
        "MB_chemo_prior": MB_chemo_prior,
        "MB_combo_prior": MB_combo_prior,

        "Avg_cost_per_month_chemo_post": Avg_cost_per_month_chemo_post,
        "Avg_cost_per_month_combo_post": Avg_cost_per_month_combo_post,
        "MB_chemo_post": MB_chemo_post,
        "MB_combo_post": MB_combo_post
    }

In [ ]:
def get_ev(
    a_value: float = a_value,
    sample_size_now: int = sample_size_now,
    sample_size_new: int = sample_size_new,
    n_psa: int = n_psa,
    seed_for_prior: int = seed,
    price_drug: float = value_based_price,
    wtp: float = wtp,
    case_params: dict = case_params,
    hr_params: dict = hr_params,
    utility: dict = utility,
    cost: dict = cost,
    cycle_range: np.ndarray = cycle_range,
    discount_factor: np.ndarray = discount_factor
) -> dict:

    # 0) some fixed
    incidence = 71794
    prevalence = 640488
    cost_per_sample = 48324.48
    uptake_rate = 0.4
    d_r = 0.03

    # 1) Get PSA results
    psa_results = compute_psa_nmb(
        a_value=a_value,
        sample_size_now=sample_size_now,
        sample_size_new=sample_size_new,
        n_psa=n_psa,
        seed_for_prior=seed_for_prior,
        price_drug=price_drug,
        wtp=wtp,
        case_params=case_params,
        hr_params=hr_params,
        utility=utility,
        cost=cost,
        cycle_range=cycle_range,
        discount_factor=discount_factor
    )

    # 2) Extract relevant MB arrays
    MB_chemo_prior = psa_results["MB_chemo_prior"]
    MB_combo_prior = psa_results["MB_combo_prior"]
    MB_chemo_post  = psa_results["MB_chemo_post"]
    MB_combo_post  = psa_results["MB_combo_post"]

    # 3) Calculate EVPI
    max_MB_prior = np.maximum(MB_chemo_prior, MB_combo_prior)
    EVPI_1 = np.mean(max_MB_prior)
    EVPI_2 = max(np.mean(MB_chemo_prior), np.mean(MB_combo_prior))
    EVPI_per_person = EVPI_1 - EVPI_2

    # 4) Calculate EVSI
    max_MB_post = np.maximum(MB_chemo_post, MB_combo_post)
    EVSI_1 = np.mean(max_MB_post)
    EVSI_2 = EVPI_2
    EVSI_per_person = EVSI_1 - EVSI_2

    # 5） Calculate ENBS
    MB_chemo_post_mean = np.mean(MB_chemo_post)
    MB_combo_post_mean = np.mean(MB_combo_post)

    ## Net Benefits during the trial (t=0,1,2; 3-year trial)
    ### half sample receive the treatment
    part1 = (sample_size_new / 2.0) * (MB_chemo_post_mean + MB_combo_post_mean)
    ### prevalent but not samples and incidence during the trial
    part2 = MB_chemo_post_mean * ((prevalence - sample_size_new)
        + np.sum(incidence * (1 / (1+d_r) ** np.arange(0, 3, 1))))

    ## Net Benefits after the trial (t=3 to t=10)
    part3 = (
    EVSI_1 * np.sum(incidence * uptake_rate * (1 / (1 + d_r) ** np.arange(3, 10, 1)))
    + MB_chemo_post_mean * np.sum(incidence * (1 - uptake_rate) * (1 / (1 + d_r) ** np.arange(3, 10, 1)))
    )

    ## Cost for the trial
    part4 = cost_per_sample * sample_size_new

    ## oppotunity cost for not conducted
    # part5 = EVSI_2 * (prevalence + np.sum(incidence * (1 / (1 + d_r) ** np.arange(0, 10, 1))))
    part5 = MB_chemo_post_mean * (prevalence + np.sum(incidence * (1 / (1 + d_r) ** np.arange(0, 10, 1))))

    ## Overall
    ENBS = (part1 + part2 + part3 - part4 - part5) / 1000000

    # 6) Enforce conditions without raising errors
    #    If any condition fails, set both to NaN
    # if (
    #     np.isnan(EVPI_per_person) or np.isnan(EVSI_per_person) or  # either is NaN
    #     (EVPI_per_person <= 0) or (EVSI_per_person <= 0) or        # must be strictly positive
    #     (EVPI_per_person <= EVSI_per_person)                  # must be strictly > EVSI
    # ):
    #     EVPI_per_person = np.nan
    #     EVSI_per_person = np.nan

    # 7) Population-level EVPI and ENBS in 10-year incidence
    EVPI = EVPI_per_person * np.sum(incidence * (1 / (1 + d_r) ** np.arange(0, 10, 1))) / 1000000
    EVSI = EVSI_per_person * np.sum(incidence * (1 / (1 + d_r) ** np.arange(0, 10, 1))) / 1000000

    # 8) Return the results (NaNs if conditions not met)
    return {
        "EVPI_per_person": EVPI_per_person,
        "EVSI_per_person": EVSI_per_person,
        "EVPI": EVPI,
        "EVSI": EVSI,
        "ENBS": ENBS
    }

In [ ]:
# a value from 0.5 to 1.0 by 0.1
a_value_arr = np.arange(0.5, 1.01, 0.1)

# price range
price_drug_arr = np.sort(np.append(np.arange(10, 20.5, 0.5), value_based_price))

# sample_size_new range
sample_size_new_arr = np.sort(np.append(np.arange(300, 1100, 100), sample_size_new))

In [ ]:
!pip install tqdm joblib tqdm_joblib
import itertools
from joblib import Parallel, delayed
from tqdm.notebook import tqdm
from tqdm_joblib import tqdm_joblib
import multiprocessing

def run_single_combination(a_val, drug_price, ss_new):
    """Runs a single combination and returns the computed EV values."""

    out = get_ev(
        a_value=a_val,
        price_drug=drug_price,
        sample_size_new=ss_new
    )

    return {
        "a_value": a_val,
        "price_drug": drug_price,
        "sample_size_new": ss_new,
        "EVPI_pp": out["EVPI_per_person"],
        "EVSI_pp": out["EVSI_per_person"],
        "EVPI": out["EVPI"],
        "EVSI": out["EVSI"],
        "ENBS": out["ENBS"]
    }

def run_grid_ev_parallel(
    a_value_arr,
    price_drug_arr,
    sample_size_new_arr
) -> pd.DataFrame:
    """
    Evaluates all combinations of (a_value, price_drug, sample_size_new) in parallel,
    displaying a progress bar, and returns a DataFrame of EVPI/EVSI results.
    """

    n_jobs = multiprocessing.cpu_count() - 1
    # n_jobs = 1

    # 1) Generate the Cartesian product of the three arrays
    all_combinations = list(
        itertools.product(a_value_arr, price_drug_arr, sample_size_new_arr)
    )
    total_combos = len(all_combinations)

    # 2) Use tqdm_joblib to show a progress bar as we run the parallel jobs
    with tqdm_joblib(tqdm(total=total_combos, desc="Computing EVPI & EVSI & ENBS")):
        results = Parallel(n_jobs=n_jobs)(
            delayed(run_single_combination)(a_val, drug_price, ss_new)
            for (a_val, drug_price, ss_new) in all_combinations
        )

    # 3) Convert list of dicts to a DataFrame
    df = pd.DataFrame(results)
    return df

/usr/local/lib/python3.11/dist-packages/tqdm_joblib/__init__.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [ ]:
# Run the parallel analysis
df_evpi_evsi = run_grid_ev_parallel(a_value_arr, price_drug_arr, sample_size_new_arr)

# Inspect the results
df_evpi_evsi.head()

Computing EVPI & EVSI & ENBS:   0%|          | 0/1188 [00:00<?, ?it/s]

  0%|          | 0/1188 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# export path
enbs_paths = "/content/drive/MyDrive/Colab Notebooks/01_Research/01_Local_Confirmatory_RCT/03_output/03_enbs"
# export it to a CSV file
df_evpi_evsi.to_csv(f"{enbs_paths}/value_price.csv", index=False)